## Stages 4

### Stages
1. Data Acquisition
2. Audio Engineering
3. Sentence Generation
4. TTS Generation 
5. Quality Control

Bash Script for running Stage 4 is isolated to a different runtime(see Colab_sateg4.ipynb) because of a conflicating version of transformers. Indic F5 requires trasnformer version -4.49.0

Note: 
- Each stage is resumable and cached in case if the colab session drops abruptly.
- You can also run stages in correct order but in differnt colab sessions. 
- All progress of each stage are cached in the mounted drive-storage. 

## Step 1 - Mount Google Drive 

Run the below cell and follow instruction on screen

In [1]:
# Mount google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# The code Repo, Clone/pull 
REPO_URL = 'https://github.com/rahulkolayikkath/synthetic-data-pipeline.git'  
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only

# Code import to working dir
!pip install -q -e .

# OUT_Dir  
OUT = '/content/drive/MyDrive/indic_synth/out_representative_sample'
os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive
/content
Cloning into 'synthetic-data-pipeline'...
remote: Enumerating objects: 391, done.
remote: Counting objects: 100% (391/391), done.
remote: Compressing objects: 100% (255/255), done.
remote: Total 391 (delta 175), reused 328 (delta 112), pack-reused 0 (from 0)
Receiving objects: 100% (391/391), 5.55 MiB | 11.45 MiB/s, done.
Resolving deltas: 100% (175/175), done.
/content/synthetic-data-pipeline
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for indic-synth (pyproject.toml) ... done


## Step 2 - Hugging Face Access 

Create a Hugging face token with read access
Make sure you have access to the following gated modelsa and datasets. Accept their terms on hugging face before you use them. 
1. ai4bharat/Kathbath 
2. ai4bharat/IndicF5
3. ai4bharat/indic-conformer-600m-multilingual
4. google/gemma-3-12b-it
5. sentence-transformers/LaBSE
6. speechbrain/spkrec-ecapa-voxceleb

In [2]:
# HF token
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token:')
login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Add HF Cache for Gemma(24gb) to avoid redownload on re-runs 
import os
os.environ["HF_HOME"] = "/content/hf_cache"

## Step 3 - Run stages

Config deatils 
1. config.representative.yaml 
- 3 languages x 2 speakers = 6 distinct speakers (gender-balanced)
- 3 languages x 1 topic x 4 sentance-types  = 12 Sentence bucket
- 8 sentances from each bucket, 8 x 12 = 96 sentences

In [4]:
# installs 
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q "transformers==4.49.0" torch torchaudio soundfile speechbrain jiwer pyyaml

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 66.2 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.0/113.0 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.

In [5]:
!python scripts/run.py --config config.representative.yaml --stages tts_generation

21:13:32 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_representative_sample seed=1234
21:13:32 INFO    run | =========== stage: tts_generation ===========
config.json: 100% 350/350 [00:00<00:00, 2.42MB/s]
model.py: 5.77kB [00:00, 22.1MB/s]
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
A new version of the following files was downloaded from https://huggingface.c